# 09 — Export Feathers with audited clocks

**You will:** freeze an export profile, run PLEB's native exporter from a host terminal,
and check clock provenance, independent Tempo2 results and repeated serialization.
This uses both the PLEB and Discovery SIFs. Model discovery here means PLEB's timing-model
search; Discovery is the separate package that reads/writes the analysis Feather format.

Run 00–08 first. Feather is a storage format, not a new fit or a QC decision. The native
exporter refits the existing terms to convergence, saves the model separately and checks
the saved-PAR residuals. It runs no new model search, QC cuts or noise-model selection.
Do not add EFAC/EQUAD/ECORR merely to make reduced chi-square look acceptable.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
replay = json.loads((ROOT/'replay_recipe.json').read_text())
host_root = Path(os.environ['PLEB_LEARNING_HOST_ROOT'])
host_source = Path(os.environ['PLEB_LEARNING_HOST_SOURCE'])
assert os.environ.get('PLEB_LEARNING_DISCOVERY_IMAGE'), (
    'Set PLEB_DISCOVERY_IMAGE to discovery_dr3_upstream.sif and relaunch learn.sh')
images = dict(pleb=os.environ['PLEB_LEARNING_HOST_IMAGE'],
              discovery=os.environ['PLEB_LEARNING_DISCOVERY_IMAGE'])
image_hashes = dict(pleb=(ROOT/'runtime/container.sha256').read_text().split()[0],
                   discovery=os.environ['PLEB_LEARNING_DISCOVERY_SHA256'])
display(pd.DataFrame([dict(image=k,path=v,sha256=image_hashes[k]) for k,v in images.items()]))

## Synthetic observatory fixture and clock policy

The training TOAs were generated at artificial station `train`. Its coordinates are
part of the simulation, so copy that recorded observatory definition into this private
export runtime. Keep the standard container runtime for everything else. This fixture
preparation is specific to synthetic data; production stations require approved records.

Dataset `clockfiles/` overlays container clocks. The synthetic `train2utc.clk` explicitly
defines a zero station-to-UTC correction because we generated UTC station times.
Our extra UTC policy contains **no stations**. For real data, only add a zero station
correction when the provider confirms that those times are already UTC; preserve the
ordinary UTC→TAI→TT chain. A missing clock must never silently become a general zero policy.

In [ ]:
export = ROOT/'native-export'
assert not export.exists(), 'Use a new export directory; do not overwrite a completed report'
runtime = export/'runtime/tempo2'
runtime.mkdir(parents=True)
stock = Path('/opt/tempo2/share/tempo2')
for entry in stock.iterdir():
    target = runtime/entry.name
    if entry.name=='observatory':
        shutil.copytree(Path(os.environ['TEMPO2'])/'observatory',target)
    elif entry.name=='clock':
        shutil.copytree(entry,target)
    else:
        target.symlink_to(entry,target_is_directory=entry.is_dir())
fixture = lab.hashes(runtime/'observatory')
(ROOT/'synthetic_export_observatories.json').write_text(json.dumps(fixture,indent=2))
policy = dict(description='Synthetic UTC train station clock supplied by dataset; no extra station overrides',
              mjd_range=[50000,70000],station_clocks=[])
clock_policy = CONFIGS/'09_utc_policy.json'
clock_policy.write_text(json.dumps(policy,indent=2)+'\n')
profile = dict(jobs=1,
    input=dict(repository='../dataset-repo',commit=replay['commit'],dataset='data',
               expected_active=replay['active'],canonical_names=True),
    containers=dict(images=images,sha256=image_hashes),
    clocks=dict(policy=clock_policy.name,sha256=hashlib.sha256(clock_policy.read_bytes()).hexdigest()),
    code={name:hashlib.sha256((Path(__import__('pleb').__file__).parent/name).read_bytes()).hexdigest()
          for name in ['residual_export.py','residual_export_worker.py','tim_utils.py',
                       'param_scan.py','resources/clock_file_trace.c']})
lab.write_config(CONFIGS/'09_feathers.toml',profile)

## Run the native PLEB command on the host

The notebook kernel is already inside a SIF. Run the printed command in a **second host
terminal**, with Apptainer and a Python environment that can import this PLEB checkout
(an editable installation, `python -m pip install -e .`, supplies its dependencies).
This avoids nested containers. Timing/Discovery calculations run inside the two pinned
images; host Python orchestrates them. The profile uses relative dataset/config paths.

Wait for completion, inspect `native-export/index.html`, then continue with the next cell.
On failure inspect `native-export/logs/`; do not change input hashes to bypass a failure.
Runtime/source changes require a newly reviewed profile and a fresh output directory.

In [ ]:
import shlex
host_command = ['env','PYTHONPATH='+str(host_source),
    'PLEB_LOG_DIR='+str(host_root/'runtime/host-logs'),
    'MPLCONFIGDIR='+str(host_root/'runtime/host-matplotlib'),
    'python','-m','pleb.residual_export',
    '--config',str(host_root/'configs/09_feathers.toml'),
    '--output-dir',str(host_root/'native-export'),'--jobs','1']
print(shlex.join(host_command))
(ROOT/'native_export_command.txt').write_text(shlex.join(host_command)+'\n')

In [ ]:
completion = export/'reports/final_summary.json'
assert completion.exists(), 'Run the host command above, then return to this cell'
summary = json.loads(completion.read_text())
assert not summary['issues'] and summary['pulsars']==len(PULSARS)
assert summary['toas']==replay['active']
assert summary['independent_container_checks']==len(PULSARS)
assert summary['independently_reproduced_feathers']==len(PULSARS)
assert not summary['model_warnings'], 'Synthetic exercise expects newly converged fits'
lab.verify_hashes(runtime/'observatory',json.loads((ROOT/'synthetic_export_observatories.json').read_text()))
clocks = json.loads((export/'reports/clock_manifest.json').read_text())
assert 'train2utc.clk' in clocks['dataset_overrides']
assert clocks['explicit_utc_files']=={}
quality = pd.read_csv(export/'reports/residual_quality.csv')
display(quality)
assert quality.fit_converged.all() and not quality.model_reused.any()
for psr in PULSARS:
    chain = json.loads((export/'reports'/f'{psr}_combined.json').read_text())['clock_chains']
    print(psr,chain)
print('Host report:',host_root/'native-export/index.html')

**Your task:** identify the dataset-supplied clock and the rest of its effective chain.
Explain why agreement between two containers and a byte-identical Feather do not prove
that residuals are white, or that every excluded TOA was bad. Scientific checks follow.